## Data Imputation Approaches


- baseline: paste in exact data from the next week
- scheduled timings:
- forward fill using avg timing between stops


In [9]:
### IMPORTS
import pandas as pd
import numpy as np
from itertools import cycle
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
import pickle

In [10]:
### LOAD ROUTE STOP SEQUENCES
with open("Datasets/route_stop_sequences.pkl", "rb") as f:
    route_stop_sequences = pickle.load(f)

In [6]:
### LOAD DATA

# Load bus data
bus_df = pd.read_csv("missing_data_imputation/bus_data_after_missing.csv")
bus_df["rt"] = bus_df["rt"].astype(str).str.strip()

# Load stops
stops_df = pd.read_csv("Datasets/stops.txt")[["stop_id", "stop_name", "stop_lat", "stop_lon"]]

# Load trips and stop_times, filtering to only routes in bus_df
trips_df = pd.read_csv("Datasets/trips.txt")[["route_id", "trip_id"]]
trips_df["route_id"] = trips_df["route_id"].astype(str).str.strip()

relevant_routes = set(bus_df["rt"])
relevant_trips = set(trips_df[trips_df["route_id"].isin(relevant_routes)]["trip_id"])

# Only load the columns we need from stop_times
stop_times_df = pd.read_csv("Datasets/stop_times.txt")[["trip_id", "stop_id"]]
stop_times_df = stop_times_df[stop_times_df["trip_id"].isin(relevant_trips)]

# Build route_id -> set of valid stop_ids
trip_to_route = trips_df.set_index("trip_id")["route_id"].to_dict()
stop_times_df["route_id"] = stop_times_df["trip_id"].map(trip_to_route)

route_to_stops = (
    stop_times_df.groupby("route_id")["stop_id"]
    .apply(set)
    .to_dict()
)

# Build cKDTree once from stops_df
stop_coords = stops_df[["stop_lat", "stop_lon"]].values
stop_tree = cKDTree(stop_coords)

def assign_stop_id(bus_lat, bus_lon, route_id, radius=0.0001):
    indices = stop_tree.query_ball_point([bus_lat, bus_lon], r=radius)

    if not indices:
        return None

    # Filter to stops actually served by this route
    valid_stops_for_route = route_to_stops.get(route_id, set())
    valid_indices = [
        i for i in indices
        if stops_df.iloc[i]["stop_id"] in valid_stops_for_route
    ]

    if not valid_indices:
        return None

    # Return nearest valid stop
    if len(valid_indices) > 1:
        candidate_coords = stop_coords[valid_indices]
        distances = np.linalg.norm(candidate_coords - np.array([bus_lat, bus_lon]), axis=1)
        nearest_idx = valid_indices[np.argmin(distances)]
    else:
        nearest_idx = valid_indices[0]

    return stops_df.iloc[nearest_idx]["stop_id"]

# Apply
bus_with_stops_df = bus_df.copy()
bus_with_stops_df["stop_id"] = bus_with_stops_df.apply(
    lambda row: assign_stop_id(row["lat"], row["lon"], row["rt"]), axis=1
)

matched = bus_with_stops_df["stop_id"].notna().sum()
total = len(bus_with_stops_df)
print(f"Matched {matched}/{total} rows ({matched/total:.1%}) to a stop")

bus_with_stops_df.head()

/var/folders/hh/2dsrbb3d021_w0zs6jdygn480000gn/T/ipykernel_41358/2562387919.py:4: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  bus_df = pd.read_csv("missing_data_imputation/bus_data_after_missing.csv")


Matched 399255/2491498 rows (16.0%) to a stop


,vid,tmstmp,lat,lon,hdg,pid,rt,des,pdist,dly,...,origtatripno,tablockid,zone,mode,psgld,stst,stsd,pulled_at,rt_chunk,stop_id
0,1095,20260220 18:41,41.878025,-87.641502,88,6351,1,34th/Michigan,2194,False,...,273982273,1 -757,NaN,1,NaN,66960,2026-02-20,2026-02-20 18:41:36 CST,"1,2,3,4,X4,N5,6,7,8,8A",NaN
1,8473,20260220 18:41,41.847638,-87.623718,178,6351,1,34th/Michigan,18066,False,...,273982272,1 -756,NaN,1,NaN,65820,2026-02-20,2026-02-20 18:41:36 CST,"1,2,3,4,X4,N5,6,7,8,8A",NaN
2,7931,20260220 18:41,41.866714,-87.624077,357,8085,1,Union Station,13113,False,...,273982246,1 -758,NaN,1,NaN,66360,2026-02-20,2026-02-20 18:41:36 CST,"1,2,3,4,X4,N5,6,7,8,8A",NaN
3,1082,20260220 18:41,41.874788,-87.644205,207,8085,1,Union Station,24853,False,...,273982245,1 -755,NaN,1,NaN,65100,2026-02-20,2026-02-20 18:41:36 CST,"1,2,3,4,X4,N5,6,7,8,8A",NaN
4,8692,20260220 18:41,41.883796,-87.627892,180,5528,2,Cottage Grove/60th,9162,False,...,273985972,2 -753,NaN,1,NaN,66000,2026-02-20,2026-02-20 18:41:36 CST,"1,2,3,4,X4,N5,6,7,8,8A",NaN


In [7]:
### STOP SEQUENCE + TIMINGS

def build_stop_sequence(route_id, route_to_stops, stops_df, t_avg):
    """
    Build an ordered stop sequence for a route with cumulative travel times.
    Returns a list of dicts with stop_id, lat, lon, and t_avg to next stop.
    """
    # Get all stop pairs for this route ordered by first appearance
    route_pairs = t_avg[t_avg["route_id"] == route_id][["stop_i", "stop_j", "t_avg_min"]].copy()
    
    if route_pairs.empty:
        return None
    
    # Build ordered sequence by chaining stop pairs (i -> j -> k -> ...)
    sequence = []
    first_stop = route_pairs.iloc[0]["stop_i"]
    current_stop = first_stop
    visited = set()
    
    while current_stop not in visited:
        visited.add(current_stop)
        next_row = route_pairs[route_pairs["stop_i"] == current_stop]
        
        if next_row.empty:
            break
        
        next_row = next_row.iloc[0]
        stop_info = stops_df[stops_df["stop_id"] == current_stop]
        
        if stop_info.empty:
            current_stop = next_row["stop_j"]
            continue
        
        sequence.append({
            "stop_id": current_stop,
            "lat": stop_info.iloc[0]["stop_lat"],
            "lon": stop_info.iloc[0]["stop_lon"],
            "t_avg_to_next_min": next_row["t_avg_min"]
        })
        current_stop = next_row["stop_j"]
    
    return sequence

In [4]:
### FORWARD FILL


def forward_fill_gap(bus_df, route_matrices, t_avg, stops_df, freq_seconds=120):
    """
    For each vehicle, detect gaps and forward fill using stop sequences.
    """
    bus_df = bus_df.copy()
    bus_df["tmstmp"] = pd.to_datetime(bus_df["tmstmp"])
    bus_df = bus_df.sort_values(["vid", "tmstmp"]).reset_index(drop=True)
    
    imputed_rows = []
    gap_threshold = pd.Timedelta(seconds=freq_seconds * 2)  # gap if > 2x normal interval
    
    for (vid, rt), group in bus_df.groupby(["vid", "rt"]):
        group = group.sort_values("tmstmp").reset_index(drop=True)
        
        # Build stop sequence for this route
        stop_sequence = build_stop_sequence(rt, route_to_stops, stops_df, t_avg)
        if stop_sequence is None or len(stop_sequence) == 0:
            continue
        
        for k in range(len(group) - 1):
            row_before = group.iloc[k]
            row_after = group.iloc[k + 1]
            
            t_before = row_before["tmstmp"]
            t_after = row_after["tmstmp"]
            gap = t_after - t_before
            
            # Only impute if gap is significant
            if gap <= gap_threshold:
                continue
            
            print(f"Vehicle {vid}, route {rt}: gap of {gap} starting at {t_before}")
            
            # Find the starting stop in the sequence
            last_stop_id = row_before.get("stop_id", None)
            start_seq_idx = 0
            if last_stop_id is not None:
                for idx, s in enumerate(stop_sequence):
                    if s["stop_id"] == last_stop_id:
                        start_seq_idx = idx
                        break
            
            # Walk forward through stop sequence, generating imputed rows
            current_time = t_before + pd.Timedelta(seconds=freq_seconds)
            seq_idx = start_seq_idx
            time_budget_min = 0  # time accumulated since last stop
            
            # Cycle through stops in case bus loops the route
            stop_cycle = cycle(range(len(stop_sequence)))
            # advance cycle to starting position
            for _ in range(seq_idx + 1):
                next_seq_idx = next(stop_cycle)
            
            current_stop = stop_sequence[seq_idx]
            next_stop = stop_sequence[next_seq_idx] if next_seq_idx != seq_idx else stop_sequence[(seq_idx + 1) % len(stop_sequence)]
            
            while current_time < t_after:
                t_to_next = current_stop["t_avg_to_next_min"]
                time_budget_min += freq_seconds / 60
                
                # Interpolate position between current and next stop
                if t_to_next and t_to_next > 0:
                    frac = min(time_budget_min / t_to_next, 1.0)
                else:
                    frac = 0
                
                interp_lat = current_stop["lat"] + frac * (next_stop["lat"] - current_stop["lat"])
                interp_lon = current_stop["lon"] + frac * (next_stop["lon"] - current_stop["lon"])
                
                imputed_rows.append({
                    **row_before.to_dict(),
                    "tmstmp": current_time,
                    "lat": interp_lat,
                    "lon": interp_lon,
                    "stop_id": current_stop["stop_id"],
                    "imputed": True
                })
                
                # Advance to next stop if we've used up the travel time
                if time_budget_min >= t_to_next:
                    time_budget_min = 0
                    seq_idx = next(stop_cycle)
                    current_stop = stop_sequence[seq_idx]
                    next_seq_idx_temp = next(stop_cycle)
                    next_stop = stop_sequence[next_seq_idx_temp]
                    # put next_seq_idx_temp back by re-advancing isn't possible,
                    # so we just track current and peek ahead manually
                    next_stop = stop_sequence[(seq_idx + 1) % len(stop_sequence)]
                
                current_time += pd.Timedelta(seconds=freq_seconds)
    
    # Combine original and imputed rows
    if imputed_rows:
        imputed_df = pd.DataFrame(imputed_rows)
        result_df = pd.concat([bus_df, imputed_df], ignore_index=True)
        result_df["imputed"] = result_df["imputed"].fillna(False)
        result_df = result_df.sort_values(["vid", "tmstmp"]).reset_index(drop=True)
    else:
        result_df = bus_df.copy()
        result_df["imputed"] = False
    
    return result_df


# Run imputation
imputed_df = forward_fill_gap(bus_with_stops_df, route_matrices, t_avg, stops_df, freq_seconds=120)

# Summary
n_imputed = imputed_df["imputed"].sum()
print(f"\nImputed {n_imputed} rows across all vehicles")
print(f"Original rows: {len(bus_with_stops_df)}, Total after imputation: {len(imputed_df)}")

NameError: name 'bus_with_stops_df' is not defined